### Final Project: Fuzzy matching of airport data and deduplication


In [19]:
project_id = "cs329e-sp2025"
dataset = "fin_air_travel"
region = "us-central1"
connection_id = "vertex-connection" # BQ requires a connection to call the model in Vertex
embedding_model = "text-embedding-005"  # latest gecko embeddings model as of 04/18/25
gemini_model = "gemini-2.5-flash-preview-04-17" # latest gemini flash model as of 04/18/25

#### Part 1: Setup

##### Create the BQ datasets


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset cs329e-sp2025.fin_air_travel


##### Create a connection resource and register the latest embeddings model (`text-embedding-005`)

In [ ]:
!bq mk --connection --location=$region --project_id=$project_id \
    --connection_type=CLOUD_RESOURCE $connection_id

Connection 798706649102.us-central1.vertex-connection successfully created


In [ ]:
!bq show --connection 798706649102.us-central1.vertex-connection

Connection 798706649102.us-central1.vertex-connection

                     name                      friendlyName   description    Last modified         type        hasCredential                                            properties                                            
 -------------------------------------------- -------------- ------------- ----------------- ---------------- --------------- ----------------------------------------------------------------------------------------------- 
  798706649102.us-central1.vertex-connection                                13 Apr 22:14:41   CLOUD_RESOURCE   False           {"serviceAccountId": "bqcx-798706649102-xf2e@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}  



##### Before running the next cell, be sure to grab the service account associated with your connection from the previous output. Then modify the command below, replacing the service account with yours.
##### Alternatively, instead of running the cell below, you can also go into the IAM console and grant your service account the "Vertex AI User" role.

In [ ]:
!gcloud projects add-iam-policy-binding $project_id --member='serviceAccount:bqcx-798706649102-xf2e@gcp-sa-bigquery-condel.iam.gserviceaccount.com' \
  --role='roles/aiplatform.user' --no-user-output-enabled

##### Replace the dataset, project, and connection as appropriate before running the next cell.

In [ ]:
%%bigquery
create or replace model fin_air_travel.embedding_model
  remote with connection `projects/cs329e-sp2025/locations/us-central1/connections/vertex-connection`
  options (endpoint = 'text-embedding-005');

Query is running:   0%|          |

""


#### Part 2: Sample the input data, `air_travel_stg.airports`


In [ ]:
%%bigquery
select * from air_travel_stg.airports order by airport_name

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,13174,American and Normandale Stn,Edina,United States,None,None,44.85701900000000000000000000000000000000,-93.35152600000000000000000000000000000000,828,-6,A,None,None,None,openflights,2025-01-24 18:55:03.442905+00:00
1,11424,Bitterfeld Bahnhof,Bitterfeld-Wolfen,Germany,None,None,51.62354600000000000000000000000000000000,12.31564500000000000000000000000000000000,261,1,E,None,station,User,openflights,2025-01-24 18:55:03.442905+00:00
2,9634,'s-Hertogenbosch Railway Station,'s-Hertogenbosch,Netherlands,None,None,51.69000000000000000000000000000000000000,5.29330000000000000000000000000000000000,25,1,E,Europe/Amsterdam,station,User,openflights,2025-01-24 18:55:03.442905+00:00
3,5849,(Duplicate) Playa Samara Airport,Playa Samara,Costa Rica,None,MRSR,9.87000000000000000000000000000000000000,-85.48000000000000000000000000000000000000,10,-6,U,America/Costa_Rica,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
4,11674,12APO,12 Apostles,Australia,None,None,-38.39600000000000000000000000000000000000,143.06300000000000000000000000000000000000,300,10,U,None,unknown,User,openflights,2025-01-24 18:55:03.442905+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12663,5797,Şanlıurfa Airport,Sanliurfa,Turkey,SFQ,LTCH,37.09429931640625000000000000000000000000,38.84709930419922000000000000000000000000,1483,3,E,Europe/Istanbul,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
12664,9044,Şanlıurfa GAP Airport,Sanliurfa,Turkey,GNY,LTCS,37.44566300000000000000000000000000000000,38.89559200000000000000000000000000000000,2708,3,E,Europe/Istanbul,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
12665,9820,Şırnak Şerafettin Elçi Airport,Cizre,Turkey,NKT,LTCV,37.36470000000000000000000000000000000000,42.05820000000000000000000000000000000000,2038,3,E,Europe/Istanbul,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
12666,3954,Šiauliai International Airport,Siauliai,Lithuania,SQQ,EYSA,55.89390182495117000000000000000000000000,23.39500045776367200000000000000000000000,443,2,E,Europe/Vilnius,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


#### Note: want to include `airport_name`, `city`, `country`, `icao`, `iata` in the embedding

#### Part 3: Create the embeddings

#### Create the embeddings on the airport's name, city, country, icao code, and iata code.
###### More details on the `ml.generate_embedding()`: https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-generate-embedding#text-embedding

In [20]:
%%bigquery

create or replace table fin_air_travel.airport_embeddings as (

with airport_content as (
  select airport_id, concat(airport_name, ' ', city, ' ', country, ' ', icao, ' ', iata) as content
  from air_travel_stg.airports
)

select
  airport_id,
  content,
  ml_generate_embedding_result as embedding
from
  ml.generate_embedding(
    model fin_air_travel.embedding_model,
    (select airport_id, content from airport_content where content is not null),
    struct('CLUSTERING' as task_type)
  )
)

Query is running:   0%|          |

""


##### Note: the embeddings are created on the `content` field. Having a `content` field is required when calling `ml.generate_embedding()`

In [21]:
%%bigquery
select * from fin_air_travel.airport_embeddings

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,content,embedding
0,12105,Aerodrom Arseniev Arseniev Russia BXBA XBA,"[-0.014932083897292614, 0.05478716269135475, -..."
1,13541,Gamar Malamo Airport Galela Indonesia WAEG GLX,"[-0.01840347796678543, 0.014528943225741386, 0..."
2,9759,Abaco I Walker C Airport Walker's Cay Bahamas ...,"[-0.0443672351539135, 0.0179140642285347, -0.0..."
3,10618,Borroloola Airport Borroloola Australia YBRL BOX,"[0.010402161628007889, 0.02908206544816494, -0..."
4,11090,Independence Municipal Airport Independence Un...,"[-0.04836883768439293, 0.006142734549939632, -..."
...,...,...,...
6181,2037,Woodbourne Airport Woodbourne New Zealand NZWB...,"[-0.027018215507268906, 0.004784183111041784, ..."
6182,5904,Lonorore Airport Lonorore Vanuatu NVSO LNE,"[0.0018744113622233272, 0.017934110015630722, ..."
6183,5868,Malolo Lailai Island Airport Malolo Lailai Isl...,"[0.010170869529247284, 0.013616975396871567, -..."
6184,2256,Babelthuap Airport Babelthuap Palau PTRO ROR,"[-0.02659524232149124, -0.006503221113234758, ..."


#### Part 4: Find the nearest neighbors based on cosine distance
###### More details on `vector_search()`: https://cloud.google.com/bigquery/docs/reference/standard-sql/search_functions#vector_search

In [22]:
%%bigquery
create or replace table fin_air_travel.airport_nearest_neighbors as
select query.airport_id as airport_id, base.airport_id as nearest_neighbor, distance
from
  vector_search(
    table fin_air_travel.airport_embeddings,
    'embedding',
    table fin_air_travel.airport_embeddings,
    'embedding',
    top_k => 2,
    distance_type => 'COSINE')
where query.airport_id != base.airport_id
order by distance

Query is running:   0%|          |

""


In [23]:
%%bigquery
select * from fin_air_travel.airport_nearest_neighbors
order by distance

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,nearest_neighbor,distance
0,8288,8366,0.014410
1,8366,8288,0.014410
2,2319,2304,0.037684
3,2304,2319,0.037684
4,7342,4307,0.037770
...,...,...,...
6181,9858,5753,0.385620
6182,12095,7064,0.406987
6183,13227,6246,0.409530
6184,9405,9410,0.409761


In [ ]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (8288, 8366)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,8366,Lawrence Municipal Airport,Lawrence,United States,LWC,KLWC,39.01119995000000000000000000000000000000,-95.21659851000000000000000000000000000000,833,-6,A,America/Chicago,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,8288,Lawrence Municipal Airport,Lawrence,United States,LWM,KLWM,42.71720123289999500000000000000000000000,-71.12339782710000000000000000000000000000,148,-5,A,America/New_York,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### This was a real duplicate.

In [ ]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (2319, 2304)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,2304,Fukue Airport,Fukue,Japan,FUJ,RJFE,32.66630172729492000000000000000000000000,128.83299255371094000000000000000000000000,273,9,U,Asia/Tokyo,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,2319,Fukui Airport,Fukui,Japan,FKJ,RJNF,36.14279937740000000000000000000000000000,136.22399902300000000000000000000000000000,19,9,U,Asia/Tokyo,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### This was **not** a duplicate, Fukue and Fukui are actually two different airports that sound similar in English

In [ ]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (7342, 4307)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,4307,Kaieteur International Airport,Kaieteur,Guyana,KAI,PKSA,5.17275476456000000000000000000000000000,-59.49148178100000000000000000000000000000,1520,-4,U,America/Guyana,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,7342,Kaieteur International Airport,Kaieteur Falls,Guyana,KIA,PSKA,5.16333300000000000000000000000000000000,-59.48333300000000000000000000000000000000,95,-4,U,America/Guyana,airport,User,openflights,2025-01-24 18:55:03.442905+00:00


##### This was a real duplicate (distance = 0.037).

In [1]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (2859, 2865)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,2865,Santa Bárbara del Zulia Airport,Santa Barbara,Venezuela,STB,SVSZ,8.97455024719238300000000000000000000000,-71.94325256347656000000000000000000000000,32,-4,U,America/Caracas,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,2859,Santa Bárbara de Barinas Airport,Santa Barbara,Venezuela,SBB,SVSB,7.80351400375366200000000000000000000000,-71.16571807861328000000000000000000000000,590,-4,U,America/Caracas,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### This was a false positive (distance = 0.05)

In [2]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (7519, 6357)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,6357,Zhanjiang Airport,Zhanjiang,China,ZHA,ZGZJ,21.21439900000000000000000000000000000000,110.35800200000000000000000000000000000000,125,8,U,Asia/Shanghai,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,7519,Zhijiang Airport,Zhijiang,China,HJJ,ZGCJ,27.44111111110000000000000000000000000000,109.70000000000000000000000000000000000000,882,8,U,Asia/Shanghai,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### That was another real duplicate (distance = 0.044)

In [3]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (5532, 45)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,45,Deer Lake Airport,Deer Lake,Canada,YDF,CYDF,49.21080017089844000000000000000000000000,-57.39139938354492000000000000000000000000,72,<NA>,A,America/St_Johns,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,5532,Deer Lake Airport,Deer Lake,Canada,YVZ,CYVZ,52.65579986572265600000000000000000000000,-94.06140136718750000000000000000000000000,1092,-6,A,America/Winnipeg,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### That was another real duplicate (distance = 0.0455).

In [7]:
%%bigquery
select * from air_travel_stg.airports
where airport_id in (6163, 7003)

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,airport_name,city,country,iata,icao,latitude,longitude,altitude,timezone_delta,daylight_savings_time,timezone_name,type,source,_data_source,_load_time
0,7003,Ulyanovsk Baratayevka Airport,Ulyanovsk,Russia,ULV,UWLL,54.26829910279999000000000000000000000000,48.22669982910000600000000000000000000000,463,4,N,Europe/Samara,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00
1,6163,Ulyanovsk East Airport,Ulyanovsk,Russia,ULY,UWLW,54.40100097656250000000000000000000000000,48.80270004272461000000000000000000000000,252,4,N,Europe/Samara,airport,OurAirports,openflights,2025-01-24 18:55:03.442905+00:00


##### That was **not** a real duplicate (distance = 0.0592)

##### It seems like 0.0456 would make a decent cutoff for selecting duplicate airports. So, we'll consider all nearest neighbor pairs that are <= 0.0456 apart to be duplicates. Surprisingly, this leaves us with only 6 duplicates as shown below:

In [5]:
%%bigquery
select * from fin_air_travel.airports_nearest_neighbors
where distance <= 0.0456
order by distance

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,nearest_neighbor,distance
0,8288,8366,0.014410
1,8366,8288,0.014410
2,2319,2304,0.037684
3,2304,2319,0.037684
4,7342,4307,0.037770
5,4307,7342,0.037770
6,7519,6357,0.044918
7,6357,7519,0.044918
8,5532,45,0.045508
9,45,5532,0.045508


#### Part 5: Assign unique cluster ids to the pairs of nearest neighbors which fall within our distance threshold (<= 0.0456)

In [24]:
import pandas
import pandas_gbq
from google.cloud import bigquery

input_table = f"{dataset}.airport_nearest_neighbors"
output_table = f"{dataset}.airport_clusters"

base_query = f"""select airport_id, nearest_neighbor
from {input_table} where distance <= 0.0456"""

bq_client = bigquery.Client()
rows = bq_client.query_and_wait(base_query)

cluster_id = 0
output_clusters = []
unique_clusters = set()

for row in rows:
    airport = row["airport_id"]
    nearest_neighbor = row["nearest_neighbor"]

    if airport not in unique_clusters:

        cluster_id += 1
        unique_clusters.add(nearest_neighbor)
        output_clusters.append((airport, cluster_id))
        output_clusters.append((nearest_neighbor, cluster_id))

        print(f"assigned {airport} and {nearest_neighbor} cluster_id {cluster_id}")

df = pandas.DataFrame.from_records(output_clusters, columns=['airport_id', 'cluster_id'])
print(df)

pandas_gbq.to_gbq(df, output_table, project_id=project_id, if_exists="replace")

assigned 8288 and 8366 cluster_id 1
assigned 2319 and 2304 cluster_id 2
assigned 7342 and 4307 cluster_id 3
assigned 7519 and 6357 cluster_id 4
assigned 45 and 5532 cluster_id 5
assigned 6382 and 6386 cluster_id 6
    airport_id  cluster_id
0         8288           1
1         8366           1
2         2319           2
3         2304           2
4         7342           3
5         4307           3
6         7519           4
7         6357           4
8           45           5
9         5532           5
10        6382           6
11        6386           6


100%|██████████| 1/1 [00:00<00:00, 5866.16it/s]


In [25]:
%%bigquery
select * from fin_air_travel.airport_clusters order by cluster_id

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id,cluster_id
0,8288,1
1,8366,1
2,2319,2
3,2304,2
4,7342,3
5,4307,3
6,7519,4
7,6357,4
8,45,5
9,5532,5


#### Part 6: Rank the airports within each cluster to select the ones to keep

In [44]:
import json, pandas, pandas_gbq
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

prompt = """Please verify the information for each pair of airports and return the most accurate airport record.
Return your answer in json using the schema {"airport_id" : integer}. For example, {"airport_id": 6386}
Do not include an explanation with your answer.
"""

sql = """select ac.cluster_id, a.* except (type, source, _data_source, _load_time)
from fin_air_travel.airport_clusters ac
join air_travel_stg.airports a on ac.airport_id = a.airport_id
order by cluster_id
"""

airports_keep = []
airports_discard = []

def do_inference(input_str):

    print("enter do_inference()")
    print("input_str:", input_str)

    vertexai.init(project=project_id, location=region)
    model = GenerativeModel(gemini_model)
    resp = model.generate_content([input_str, prompt])

    resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
    print("resp_text:", resp_text)

    results = json.loads(resp_text)

    return results


bq_client = bigquery.Client()
rows = bq_client.query_and_wait(sql)

airport_pairs_list = []
cluster_id = ""
prev_cluster_id = ""
combined_results = []

for row in rows:

    cluster_id = row['cluster_id']

    row_dict = {}
    row_dict["airport_id"] = row['airport_id']
    row_dict["airport_name"] = row['airport_name']
    row_dict["city"] = row['city']
    row_dict["country"] = row['country']
    row_dict["iata"] = row['iata']
    row_dict["icao"] = row['icao']
    row_dict["latitude"] = str(row['latitude'])
    row_dict["longitude"] = str(row['longitude'])
    row_dict["altitude"] = str(row['altitude'])

    airport_pairs_list.append(json.dumps(row_dict))

    if cluster_id == prev_cluster_id:
        airport_pairs_str = ",".join(airport_pairs_list)
        results = do_inference(airport_pairs_str)
        print(results)
        combined_results.append(results)
        airport_pairs_list.clear()

    prev_cluster_id = cluster_id

# write results to BQ
print("combined_results:", combined_results)
df = pandas.DataFrame(combined_results)

table_id = "fin_air_travel.airport_keep" # output table
pandas_gbq.to_gbq(df, table_id, project_id=project_id, if_exists="replace")

enter do_inference()
input_str: {"airport_id": 8366, "airport_name": "Lawrence Municipal Airport", "city": "Lawrence", "country": "United States", "iata": "LWC", "icao": "KLWC", "latitude": "39.01119995", "longitude": "-95.21659851", "altitude": "833"},{"airport_id": 8288, "airport_name": "Lawrence Municipal Airport", "city": "Lawrence", "country": "United States", "iata": "LWM", "icao": "KLWM", "latitude": "42.717201232899995", "longitude": "-71.1233978271", "altitude": "148"}
resp_text: {"airport_id": 8366}
{'airport_id': 8366}
enter do_inference()
input_str: {"airport_id": 2304, "airport_name": "Fukue Airport", "city": "Fukue", "country": "Japan", "iata": "FUJ", "icao": "RJFE", "latitude": "32.66630172729492", "longitude": "128.83299255371094", "altitude": "273"},{"airport_id": 2319, "airport_name": "Fukui Airport", "city": "Fukui", "country": "Japan", "iata": "FKJ", "icao": "RJNF", "latitude": "36.1427993774", "longitude": "136.223999023", "altitude": "19"}
resp_text: {"airport_id"

100%|██████████| 1/1 [00:00<00:00, 6132.02it/s]


In [45]:
%%bigquery
select * from fin_air_travel.airport_keep

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id
0,8366
1,2304
2,4307
3,7519
4,45
5,6382


##### Compute the list of airports to discard

In [46]:
%%bigquery
create or replace table fin_air_travel.airport_discard as
    select airport_id from fin_air_travel.airport_clusters
    except distinct
    select airport_id from fin_air_travel.airport_keep

Query is running:   0%|          |

""


In [50]:
%%bigquery
select * from fin_air_travel.airport_discard

Query is running:   0%|          |

Downloading:   0%|          |

,airport_id
0,5532
1,6386
2,7342
3,6357
4,2319
5,8288


#### Part 7: Construct the final airport table

##### Exclude the airports from the discard list

In [47]:
%%bigquery
create or replace table fin_air_travel.airport_final as
    select * from air_travel_stg.airports
    where airport_id not in (select airport_id from fin_air_travel.airport_discard)

Query is running:   0%|          |

""


##### Check for duplicates

In [49]:
%%bigquery
select icao, count(*) as count
from fin_air_travel.airport_final
group by icao
having count(*) > 1

Query is running:   0%|          |

Downloading:   0%|          |

,icao,count
0,None,4507


#### Part 8: Conclusion

##### We have constructed an embeddings-based approach to detecting duplicate airports in our table. This approach made use of text embeddings and nearest neighbor search to find the most similar airport pairs. We then used the language model to rank each pair of airports by level of accuracy. Finally, we discarded the aiports that were ranked low to come up with our final table.

##### Previously, we had detected duplicates by going a GROUP BY on the icao code. However, this method was not always accurate because we found certain records to have incorrect icao values. The embeddings method is helpful for catching those cases.